# Machine Learning Class 1: Classification & How to Tell a Good Model from a Flattering One

Welcome to your first hands-on session. We'll build a machine learning model today, and then spend most of our time learning not to trust it too quickly.

### 🎯 What You'll Learn
1. 🏷️ **Classification** - Teaching a computer to sort things into categories
2. 🪤 **The Accuracy Trap** - Why "92% accurate" can mean "useless"
3. 🔲 **The Confusion Matrix** - The four numbers every other metric is built from
4. ⚖️ **Precision, Recall & F1** - *Which* mistakes does the model make?
5. 🎚️ **The Threshold** - The dial nobody tells you you're allowed to turn
6. 📈 **ROC & AUC** - Judging a model across *every* possible threshold
7. 🔁 **Cross-Validation** - How to stop fooling yourself with a lucky split
8. 🧭 **Sampling Bias** - The failure no algorithm can fix

### 🤖 **The Big Idea**

Traditional programming means writing the rules yourself:

> `if temperature > 38 and cough == True: diagnosis = "flu"`

That works right up until the rules get complicated. Machine learning flips it around: instead of writing the rules, you show the computer **examples with known answers** and let it work out the rules itself. That's **supervised learning** — supervised because every training example comes with the right answer attached.

When the answer is a **category** ("spam or not", "which species", "disease or healthy"), the task is **classification**. When the answer is a **number** ("what price", "how much biomass"), it's **regression**, which is Class 2.

Classification comes in a few flavours:
- **Binary** - Two categories. Spam / not spam. That's our setting today.
- **Multiclass** - One label out of many. Which of 200 bird species is in this photo?
- **Multilabel** - Several labels at once. This article is about *both* politics *and* economics.

### 🩺 **Today's Problem**

We're building a screening tool for a disease. It takes a few routine measurements and predicts whether a patient should be sent for further tests.

The two kinds of mistake here are obviously not equally bad, and by the end of this notebook you'll be able to say exactly how much you're trading one for the other.

Let's go. 🚀

In [ ]:
# If running on Google Colab, clone the repo (if needed),
# move into the repo directory, and ensure it’s on the Python path.

import sys, os

def in_colab():
    try: import google.colab; return True
    except: return False

if in_colab():
    repo = "Hands-On-Notebooks"
    if os.path.basename(os.getcwd()) != repo:
        if not os.path.exists(repo):
            !git clone https://github.com/BridgingAISocietySummerSchools/{repo}
        %cd {repo}
    if '.' not in sys.path:
        sys.path.append('.')

In [ ]:
# Quick Setup - Import Our Tools

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Import our plotting utilities
from plotting_utils.classification import (
    generate_screening_data,
    make_classifier,
    plot_class_balance,
    plot_confusion_matrix,
    plot_metric_bars,
    create_threshold_interactive,
    plot_threshold_sweep,
    plot_pr_curve,
    plot_roc_curves,
    plot_split_lottery,
    plot_kfold_diagram,
    plot_bias_demo,
)

In [ ]:
# Set a random seed so that everyone in the room sees the same numbers.
# (generate_screening_data takes its own `seed` argument, so it is reproducible
#  regardless of this line.)
np.random.seed(42)

## Part 1: The Data — 10,000 Patients

### 🏥 The Screening Study

Three clinics have pooled their records. For every patient we have six routine measurements and, crucially, whether they **actually turned out to have the disease** — established later by a proper diagnostic test.

Those known answers are what makes this *supervised* learning. Without them there's nothing to learn from.

| Column | What it is |
|---|---|
| `age` | Patient age in years |
| `bmi` | Body mass index |
| `family_history` | 1 = a close relative has had the disease |
| `smoker` | 1 = current smoker |
| `marker_a` | An expensive blood test |
| `marker_b` | A cheap blood test |
| `site` | Which clinic recruited them (A, B or C) |
| `has_disease` | **The answer we want to predict.** 1 = yes |

Note that `site` isn't a medical measurement — it's a fact about how the data was collected. Keep it in the back of your mind: we come back to it at the very end, and it turns out to matter more than any of the blood tests.

In [ ]:
# Generate the dataset
data, feature_cols = generate_screening_data(n_samples=10_000)

display(data.head(10))
print(f"Patients: {len(data):,}")
print(f"Features we will give the model: {feature_cols}")

### ⚖️ How Many Patients Actually Have the Disease?

Before building anything, always look at this. It decides how you have to read every number that follows.

In [ ]:
plot_class_balance(data['has_disease'])

🔍 **About one patient in ten.** That's what makes this a realistic screening problem — and a treacherous one. Real screening programmes are often far more lopsided (1 in 100, 1 in 1,000), and everything you're about to see gets *worse* as the imbalance grows.

## Part 2: 🪤 The Accuracy Trap

Here's a machine learning model. It took no effort to build, it uses none of the measurements, and it scores over 90%.

Its entire strategy: **say "healthy" to everybody.**

In [ ]:
X = data[feature_cols]
y = data['has_disease']

# The laziest possible "model": always predict the most common class.
lazy = DummyClassifier(strategy='most_frequent').fit(X, y)
lazy_predictions = lazy.predict(X)

print(f"🪤 Accuracy of the always-say-'healthy' model: {accuracy_score(y, lazy_predictions):.1%}")
print()
print(f"   Patients who actually have the disease: {y.sum():,}")
print(f"   Patients it correctly identified:       {int(((lazy_predictions == 1) & (y == 1)).sum()):,}")

### 💀 Ninety percent accurate. Zero patients helped.

This is the answer to *"why does a model with 99% accuracy sometimes tell us almost nothing useful?"* Accuracy counts how often the model is right. When one answer is right 90% of the time by default, always giving that answer is 90% accurate and worthless.

Two consequences:

1. **An accuracy number without a baseline is not a result.** "92% accurate" means nothing until you know what saying nothing at all would have scored. Here, that's 90%.
2. **Accuracy treats all mistakes as equal.** Telling a healthy person "come back for more tests" and telling a sick person "you're fine" both count as one error. In a hospital they are not remotely the same thing.

Everything else in this notebook exists because of these two problems.

## Part 3: ✂️ A Real Model, Judged Honestly

### Why we can't test on the training data

We must **not** measure a model on the same data it learned from. That's like studying with the answer key in front of you and then declaring yourself a genius when you ace that exact exam. 🤓

So we split the patients in two:

- **Training set (70%)** - The model learns from these.
- **Test set (30%)** - Locked away, used only at the end. These stand in for future patients the model has never met.

`stratify=y` keeps the same 10% disease rate in both halves. Without it, a random split could hand us a test set with barely any cases at all.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training patients: {len(X_train):,}  ({y_train.sum():,} with the disease)")
print(f"Test patients:     {len(X_test):,}  ({y_test.sum():,} with the disease)")

### 🤖 Our Model: Logistic Regression

We'll use **logistic regression** throughout. For today, treat it as a black box with one useful property:

> Given a patient's measurements, it outputs a **probability**: a number between 0 and 1 saying how likely that patient is to have the disease.

That probability is what we'll spend the rest of the notebook squeezing. (Despite the name it's a classifier, not a regression model, though it is a close cousin of the linear regression you'll meet in Class 2.)

In [ ]:
model = make_classifier()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
model_accuracy = accuracy_score(y_test, predictions)

lazy_accuracy = accuracy_score(y_test, DummyClassifier(strategy='most_frequent')
                               .fit(X_train, y_train).predict(X_test))

print(f"🪤 Always-say-'healthy' baseline: {lazy_accuracy:.1%}")
print(f"🤖 Our trained model:             {model_accuracy:.1%}   ({model_accuracy - lazy_accuracy:+.1%})")

### 🚩 Read That Again

**2.6 percentage points.** All that machinery, six measurements per patient, and the headline number barely moves off "say healthy to everyone".

So is the model bad? **We cannot tell from this number.** Accuracy collapses four different quantities into one, and the interesting differences are hiding inside. Let's look at the four numbers it collapsed.

## Part 4: 🔲 The Confusion Matrix

Every binary prediction lands in one of four boxes. That's the whole idea.

|  | Model says **healthy** | Model says **disease** |
|---|---|---|
| **Actually has disease** | ❌ False Negative - *a missed case* | ✅ True Positive - *caught* |
| **Actually healthy** | ✅ True Negative - *correctly cleared* | ❌ False Positive - *a false alarm* |

The two ✅ boxes are correct predictions. The two ❌ boxes are mistakes, and they're **completely different mistakes**. One sends a healthy person for an unnecessary scan. The other sends a sick person home.

In [ ]:
tn, fp, fn, tp = plot_confusion_matrix(y_test, predictions,
                                       title="Our model on the 3,000 held-out patients")

### 😬 Where the 92.6% Came From

The model correctly cleared almost every healthy patient. Since healthy patients are 90% of everybody, that alone nearly guarantees a high accuracy.

But look at the top row. Of the 300 patients who really had the disease, the model **missed 179 of them**. It found fewer than half.

So we have a screening tool that lets most of the disease walk out of the door, and accuracy called it a 92.6% success.

## Part 5: ⚖️ Precision, Recall & F1

The four boxes are the raw material. These three numbers are how people actually talk about them, and each answers a different question.

- **Precision** - *When the model raises the alarm, how often is it right?* Of everyone we flagged, what fraction really had the disease? **Low precision = lots of false alarms.**
- **Recall** - *Of all the people who really had the disease, how many did we catch?* **Low recall = lots of missed cases.**
- **F1** - Their harmonic mean, a single number balancing the two. Useful for ranking models, but it hides the trade-off you may actually care about.

Let's compute them straight from the four boxes.

In [ ]:
precision = tp / (tp + fp)          # of those we flagged, how many were right?
recall    = tp / (tp + fn)          # of the real cases, how many did we find?
f1        = 2 * precision * recall / (precision + recall)

print(f"Flagged for follow-up: {tp + fp:,} patients")
print(f"   ...of whom really had the disease: {tp:,}")
print()
print(f"⚖️ Precision: {precision:.1%}  — {precision:.0%} of our alarms are real")
print(f"🎣 Recall:    {recall:.1%}  — we catch {recall:.0%} of the actual cases")
print(f"🎯 F1:        {f1:.3f}")

In [ ]:
# sklearn computes the same thing, plus the numbers for the 'healthy' class.
print(classification_report(y_test, predictions,
                            target_names=['Healthy', 'Has disease'], digits=3))

🔍 **Look at how lopsided that report is.** The `Healthy` row scores beautifully (precision 0.937, recall 0.984). The `Has disease` row — the row we built this thing for — has recall 0.403.

The model has learned the safest possible habit: when in doubt, say healthy. `support` on the right shows why that pays off, with 2,700 healthy patients against 300 sick ones.

In [ ]:
plot_metric_bars(
    metrics={'Accuracy': model_accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1},
    baseline={'Accuracy': lazy_accuracy, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0},
    title="Our model vs. always saying 'healthy'",
)

### 💡 Choosing a Metric Is a Values Decision

Notice that the grey baseline **ties us on accuracy** and scores zero on everything else. Accuracy was the one metric that couldn't tell the difference between our model and a model that does nothing.

Which metric you report depends entirely on what the mistakes cost:

- **A spam filter** should optimise **precision**. Missing a spam email is a mild annoyance. Silently binning a job offer isn't. Never flag something unless you're sure.
- **A cancer screen** should optimise **recall**. A false alarm costs an anxious week and a second test. A missed case can cost a life. Flag anything remotely suspicious.

Same mathematics, opposite decision. The maths cannot tell you which one you want. It is a judgement about human consequences, and it belongs to the people who understand the domain rather than the people who understand the algorithm.

Our screening tool has recall of 40%. That's the wrong trade for a screening tool. Can we fix it?

## Part 6: 🎚️ The Threshold Is a Choice, Not a Law

The model never actually says "disease" or "healthy". It outputs a **probability**, and then somebody compares it to **0.5**.

That 0.5 is not a law of nature. It's a default, and you're allowed to change it.

In [ ]:
# What the model really produces, before anyone rounds it off:
risk_scores = model.predict_proba(X_test)[:, 1]

peek = pd.DataFrame({
    'predicted risk': risk_scores[:8].round(3),
    'flagged at 0.5?': np.where(risk_scores[:8] >= 0.5, 'yes', 'no'),
    'actually has disease': y_test.values[:8],
})
display(peek)

print(f"Predicted risk ranges from {risk_scores.min():.3f} to {risk_scores.max():.3f}")

### 🎮 Interactive: Turn the Dial

Move the threshold and watch what happens. Two things to try:

- 🎣 **Drag it down to 0.10.** How many more real cases do we catch? What's the cost?
- 🎯 **Drag it up to 0.80.** How much do you trust a flag now? How many patients did we abandon?

In [ ]:
create_threshold_interactive(y_test, risk_scores)

### 🔍 What You Should Have Seen

Nothing about the *model* changed while you moved that slider — not one coefficient. All that moved was where we drew the line.

- Lower the threshold → catch more real cases (**recall up**), raise more false alarms (**precision down**).
- Raise the threshold → alarms become trustworthy (**precision up**), more sick patients go home undiagnosed (**recall down**).

That's the **precision-recall trade-off**, and there's no free lunch. Let's see the whole trade-off at once instead of one setting at a time.

In [ ]:
best_threshold = plot_threshold_sweep(y_test, risk_scores)
print(f"If we simply maximise F1, the best threshold is {best_threshold:.2f}, not 0.50.")

In [ ]:
plot_pr_curve({'Our model': (y_test, risk_scores)}, prevalence=y_test.mean())

### 📉 Reading the Precision-Recall Curve

Each point on that curve is one threshold. Sliding along it is exactly what you were doing with the slider.

The dotted line at 10% is the floor: flag *every* patient and you get perfect recall with 10% precision, because 10% of patients have the disease. **A useless model sits on that line.** Ours is well above it, so it has genuinely learned something even though its accuracy was barely better than the lazy baseline.

## Part 7: 📈 ROC & AUC — Judging Every Threshold at Once

The model's quality depends on a threshold, and the threshold depends on what you care about. So how do you compare two models *before* deciding where to draw the line?

You look at all thresholds at once. That's the **ROC curve**:

- **x-axis, false positive rate:** Of the healthy patients, what fraction did we alarm?
- **y-axis, true positive rate:** Of the sick patients, what fraction did we catch? (Recall again, under a different name.)

Sweep the threshold from 1 to 0 and trace the point. A model that ranks sick patients above healthy ones climbs steeply into the **top-left corner**. A model that has learned nothing produces the **diagonal**: to catch 60% of cases it must alarm 60% of healthy people, which is what coin-flipping gets you.

**AUC** is the area under that curve, squeezed into one number: **1.0 = perfect · 0.5 = coin flip.**

To make this concrete, we train a second, deliberately handicapped model. Same algorithm, but it only gets to see the **cheap blood test** (`marker_b`).

In [ ]:
# Same algorithm, same training patients -- only the information differs.
cheap_model = make_classifier().fit(X_train[['marker_b']], y_train)
cheap_scores = cheap_model.predict_proba(X_test[['marker_b']])[:, 1]

aucs = plot_roc_curves({
    'Full workup (all six measurements)': (y_test, risk_scores),
    'Cheap blood test only (marker_b)':   (y_test, cheap_scores),
}, title="Two models, same algorithm, different information")

### 🔍 What the Two Curves Tell Us

- **The full model (AUC 0.887).** Pick a sick patient and a healthy patient at random: there's an 89% chance the model gives the sick one the higher risk score. AUC measures exactly that — the probability of ranking them the right way round.
- **The cheap test alone (AUC 0.549).** Barely off the diagonal. On its own, that blood test does not separate sick from healthy.

Nothing about the *algorithm* changed between those two curves. Only the data did. Part 9 is entirely about that.

⚠️ **One caveat.** ROC curves are flattering when classes are imbalanced. The false-positive rate has 2,700 healthy patients in its denominator, so 43 false alarms look like a rounding error, while precision divides by the 164 patients we actually flagged and feels them immediately. At 1-in-1,000 prevalence a model can post a gorgeous AUC and still be useless in the clinic. When positives are rare, **look at the precision-recall curve too.**

## Part 8: 🔁 Cross-Validation — Are We Sure?

Every number so far came from **one** train/test split, chosen by `random_state=42`. How much of that 0.887 was the model, and how much was luck in which 3,000 patients landed in the test set?

Let's re-run the whole thing 30 times, changing nothing but the random seed.

In [ ]:
split_aucs = plot_split_lottery(X, y, n_repeats=30)
print(f"Spread between the luckiest and unluckiest split: {split_aucs.max() - split_aucs.min():.3f} AUC")

### 🎲 Four Points of AUC, for Free

Same data, same model, same code. The only thing that changed was which patients happened to land in the test set, and the score wanders by about four points of AUC.

So if one paper reports 0.91 AUC and another reports 0.88, you've learned **nothing** about which model is better. That gap is smaller than the noise in a single split.

The fix is **k-fold cross-validation**. Cut the data into k equal slices, train on k−1 of them, test on the one held back, then rotate so every slice gets its turn. Average the k scores.

Think of five practice exams, each holding back a different set of questions: nobody's score rests on which questions they happened to be lucky with.

In [ ]:
plot_kfold_diagram(n_splits=5)

In [ ]:
cv_scores = cross_val_score(
    make_classifier(), X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
)

print("AUC on each of the 5 folds:", np.round(cv_scores, 3))
print()
print(f"📊 Cross-validated AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

### ✅ Why This Is the Number to Report

1. **Every patient gets tested on exactly once.** No data is wasted, and the estimate is not hostage to one lucky draw.
2. **You get an error bar.** `± 0.010` is not decoration. It's the honest statement that your model is somewhere around 0.89 rather than exactly at it, and any comparison between two models whose intervals overlap is not a comparison at all.

`StratifiedKFold` keeps the 10% disease rate inside every fold, which matters here: with plain k-fold, a fold could end up with far too few cases to score meaningfully.

If you've used leave-one-out validation in ecology or statistics, this is the general form of the same idea.

## Part 9: 🧭 The Failure No Algorithm Can Fix

We've been careful. We held out data, we cross-validated, we looked past accuracy. Our model gets 0.89 AUC and we'd happily put that in a paper.

Now suppose the study had been run slightly differently.

Remember the `site` column, the clinic each patient was recruited at? We never gave it to the model, and it has no effect on the disease whatsoever. But the three clinics see very different people.

In [ ]:
site_summary = data.groupby('site').agg(
    patients=('age', 'size'),
    mean_age=('age', 'mean'),
    disease_rate=('has_disease', 'mean'),
)
display(site_summary.round(3))

**Site A is a young-adults clinic.** Its patients average 38, and only 2% of them have the disease. Sites B and C see progressively older, sicker populations.

Now imagine an entirely realistic scenario: the research grant only covered **one** clinic. The study ran at Site A, recruited 2,441 patients, and built a model. Same algorithm, same careful train/test discipline, same everything — just a different set of people walking through the door.

Then the tool gets deployed nationally, on the **55+ screening population** that screening programmes actually target.

In [ ]:
train_records = data.loc[X_train.index]
test_records = data.loc[X_test.index]

# The study that only ran at Site A
site_a_patients = train_records[train_records['site'] == 'A']
site_a_model = make_classifier().fit(site_a_patients[feature_cols],
                                     site_a_patients['has_disease'])

# Deployment reality: patients aged 55 and over, from all three sites
older_patients = test_records[test_records['age'] >= 55]
y_older = older_patients['has_disease']

auc_site_a = roc_auc_score(y_older, site_a_model.predict_proba(older_patients[feature_cols])[:, 1])
auc_all_sites = roc_auc_score(y_older, model.predict_proba(older_patients[feature_cols])[:, 1])

print(f"Site A study:  {len(site_a_patients):,} patients, "
      f"{site_a_patients['has_disease'].sum():,} of them with the disease")
print(f"Deployed on:   {len(older_patients):,} patients aged 55+ "
      f"({y_older.mean():.1%} disease rate)")
print()
print(f"🧭 Model trained at Site A only:  AUC {auc_site_a:.3f}")
print(f"🌍 Model trained at all sites:    AUC {auc_all_sites:.3f}")

In [ ]:
plot_bias_demo({
    'Trained at Site A only': auc_site_a,
    'Trained at all three sites': auc_all_sites,
}, title="Both models deployed on the 55+ screening population")

### 🔬 Why It Failed

Eleven points of AUC, thrown away by a recruitment decision. Let's open both models up and look at the weight each one puts on each measurement:

In [ ]:
comparison = pd.DataFrame({
    'Site A study': site_a_model[-1].coef_[0],
    'All three sites': model[-1].coef_[0],
}, index=feature_cols)
display(comparison.round(2))

Look at **`marker_a`**: the all-sites model weights it at 1.39, by far its most important input. The Site A model weights it at 0.34, essentially dismissing it. Same story for `age`: 1.21 across all sites, 0.19 at Site A.

Why? In this disease `marker_a` only becomes informative in **older** patients, and Site A saw almost nobody over 50. From inside that dataset the expensive blood test genuinely looks useless, and the model was right to conclude that *about the patients it was shown*.

So the Site A model is not badly built. It is not undertrained, overfitted, or badly tuned. It faithfully learned a real pattern that only holds in young adults, and then met the rest of the world.

No algorithm would have saved it. A random forest trained on Site A fails the same way, and so does a neural network. Cross-validation would not have caught it either, because every fold has the same blind spot.

> **A model can only learn what its data contains. Gaps in the data become blind spots in the model, and the model has no way of telling you they're there.**

This is the mechanism behind most of the AI failures you read about: facial recognition trained mostly on one demographic, a species distribution model built from museum specimens that maps where *collectors* went, a hiring model trained on who a company hired before. None of them are algorithm bugs. **The algorithm is rarely the bottleneck; the data collection is.**

## 🎉 What You've Accomplished Today!

In under an hour you've built a classifier — and learned the several distinct ways it could have fooled you.

### ✅ **Core Concepts Learned:**
1. **Classification** 🏷️ – Predicting a category from labelled examples
2. **Baselines** 🪤 – 90% accuracy from a model that helps nobody
3. **The Confusion Matrix** 🔲 – The four numbers everything else is built from
4. **Precision, Recall & F1** ⚖️ – *Which* mistakes, not just how many
5. **The Decision Threshold** 🎚️ – The dial you're allowed to turn
6. **ROC & AUC** 📈 – Comparing models across every threshold at once
7. **Cross-Validation** 🔁 – A number with an error bar, not a lottery ticket
8. **Sampling Bias** 🧭 – The failure that no algorithm can fix

### 🎯 **Key Insights:**
- **Always ask "compared to what?"** – an accuracy number without a baseline is not a result
- **Accuracy hides the mistakes that matter** – 92.6% accurate, and it missed 179 of 300 cases
- **Choosing a metric is a values decision** – the maths is identical for a spam filter and a cancer screen; the right answer isn't
- **0.5 is a default, not a law** – the threshold is yours to set, and setting it is a policy choice
- **One split is a lottery ticket** – cross-validate, and report the spread
- **A high AUC can still be a useless clinical tool** – check precision-recall when positives are rare
- **The model cannot know what it was never shown** – and it will not warn you

### 🔗 **Coming Up Next:**
- **Class 2 (Regression)**: The same fit-and-evaluate loop, but predicting a **number** instead of a category, and how a model actually *learns* by minimising its errors
- **Class 3 (Trees & Forests)**: Models that ask questions instead of drawing lines, where precision and recall come back
- **Class 4 (Neural Networks)**: The same ideas at enormous scale
- **Class 5 (Modern AI)**: What changes when a model is trained on everything at once

### 🌟 **The Big Picture:**
Everything today was one lesson: **a model's score is a claim, and claims need evidence.** Held-out data, a baseline, the right metric for the stakes, an error bar, and an honest account of who is in the training data. That habit transfers to every model in this course, and to every ML result you'll ever read.

**Well done today!** 🩺📊